In [0]:
%pip install fastf1 pyproj matplotlib --quiet

In [0]:
import fastf1
from datetime import datetime
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib
import os

# Mapping from FastF1 location names to circuit_data names
location_mapping = {
    'Miami Gardens': 'Miami',
    'Montréal': 'Montréal',  # Handle potential encoding issues
    'Montreal': 'Montréal',
    'São Paulo': 'São Paulo',
    'Sao Paulo': 'São Paulo',
    # Add more mappings as needed
}

tyre_for_each_race = {
        '345': ['Melbourne', 'Jeddah', 'Miami', 'Spielberg', 'Monza', 'Baku'],
        '234': ['Shanghai', 'Silverstone', 'Budapest', 'Zandvoort'],
        '123': ['Suzuka', 'Sakhir', 'Barcelona', 'Marina Bay', 'Austin', 'Mexico City', 'São Paulo', 'Las Vegas', 'Lusail', 'Yas Island'],
        '456': ['Imola', 'Monaco', 'Montréal'],
        '134': ['Spa-Francorchamps']
}

circuit_data = pd.DataFrame({
    'Name'              : ['Sakhir', 'Jeddah', 'Melbourne', 'Suzuka', 'Shanghai', 
                           'Miami', 'Imola', 'Monaco', 'Montréal', 'Barcelona', 
                           'Spielberg', 'Silverstone', 'Budapest', 'Spa-Francorchamps', 'Zandvoort', 
                           'Monza', 'Baku', 'Marina Bay', 'Austin', 'Mexico City', 
                           'São Paulo', 'Las Vegas', 'Lusail', 'Yas Island'],

    'CircuitLength'    : [5.412, 6.174, 5.303, 5.807, 5.451,
                          5.412, 4.909, 3.337, 4.361, 4.655,
                          4.326, 5.891, 4.381, 7.004, 4.259, 
                          5.793, 6.003, 4.94, 5.513, 4.304,
                          4.309, 6.201, 5.419, 5.281
                          ],

    'Number of Laps'    : [57, 50, 58, 53, 56, 
                           57, 63, 78, 70, 66, 
                           71, 52, 70, 44, 72, 
                           53, 51, 62, 56, 71,
                           71, 50, 57, 58
                           ],
    'NumberOfTurns'     : [15, 27, 14, 18, 16, 
                           19, 19, 19, 14, 14, 
                           10, 18, 16, 19, 14,
                           11, 20, 19, 20, 17,
                           15, 17, 16, 16],

    'AverageAngleAbs'      : [216.48826392555523, 104.29061912054225, 89.05551872226904, 176.5787662838012, 85.05921720383002, 97.73264647184526, 96.86016529526317, 92.17177042065626, 80.46619041837172, 206.491918642282, 69.25595030609429, 83.35334581360956, 82.49930645037058, 100.01189241086918, 79.15591232751017, 90.24350869943476, 88.96085343584257, 107.3090988326499, 105.37917997627748, 97.04221228936298, 84.39904822399014, 107.92673960682204, 98.32126702370911, 74.68854478807737], #using absolute value of angles
    'AverageAngle': [-216.48826392555523, -58.73678219611126, -0.38805452678854174, -176.5787662838012, -6.794049069006466, -73.75106253395187, -32.77440998822381, 13.800079131391477, -16.00914472766793, -206.491918642282, 21.23424492222545, -4.988783677148164, 12.842773471410073, 16.089674092163328, 0.8365054834529914, 51.36152104590719, -23.597804580524446, -5.6658573384283315, -58.16669648878112, -58.946680967818814, 17.059245892889503, 9.372151834849562, -7.006878086293883, -12.815079854190804],
})


In [0]:
def _process_driver_telemetry(driver, race_laps_df):
    """
    Worker function to process telemetry for a single driver (runs in separate process).
    
    Args:
        driver: Driver code (e.g., 'HAM', 'VER')
        race_laps_df: DataFrame containing all race laps
    
    Returns:
        dict with 'driver', 'features' (list of dicts), 'processed', 'skipped'
    """
    import numpy as np
    
    features = []
    processed = 0
    skipped = 0
    
    try:
        # Get laps for this driver only
        driver_laps = race_laps_df[race_laps_df['Driver'] == driver]
        
        if len(driver_laps) == 0:
            return {'driver': driver, 'features': [], 'processed': 0, 'skipped': 0}
        
        # Process each lap
        for idx, lap in driver_laps.iterrows():
            lap_num = lap['LapNumber']
            try:
                # Get telemetry for this specific lap
                lap_telemetry = lap.get_telemetry()
                
                if lap_telemetry is None or len(lap_telemetry) < 10:
                    skipped += 1
                    continue
                
                # Extract arrays
                throttle = lap_telemetry['Throttle'].values
                brake = lap_telemetry['Brake'].values
                speed = lap_telemetry['Speed'].values
                distance = lap_telemetry['Distance'].values
                
                # 1. THROTTLE COMMITMENT - % time at full throttle (>95%)
                wot_mask = throttle > 95
                throttle_commitment = wot_mask.sum() / len(throttle) if len(throttle) > 0 else 0
                
                # 2. LIFT-AND-COAST - Distance between throttle close and brake on
                brake_on = np.where(np.diff(brake.astype(int)) > 0)[0]
                
                lac_distances = []
                for brake_idx in brake_on:
                    if brake_idx < 5:
                        continue
                    
                    throttle_before = throttle[max(0, brake_idx-50):brake_idx]
                    throttle_close_points = np.where(throttle_before < 5)[0]
                    
                    if len(throttle_close_points) > 0:
                        last_throttle_close_idx = throttle_close_points[-1] + max(0, brake_idx-50)
                        
                        if last_throttle_close_idx < brake_idx and brake_idx < len(distance):
                            coast_dist = distance[brake_idx] - distance[last_throttle_close_idx]
                            if coast_dist > 0:  # Sanity check
                                lac_distances.append(coast_dist)
                
                avg_lac = np.mean(lac_distances) if len(lac_distances) > 0 else 0
                
                # 3. BRAKE INTENSITY - Average deceleration rate
                brake_mask = brake > 0
                
                if brake_mask.sum() > 1:
                    speed_change = -np.diff(speed)
                    speed_change_braking = speed_change[brake_mask[:-1]]
                    avg_brake_intensity = np.mean(speed_change_braking) if len(speed_change_braking) > 0 else 0
                else:
                    avg_brake_intensity = 0
                
                # Store features for this lap
                features.append({
                    'LapNumber': lap_num,
                    'ThrottleCommitment': throttle_commitment,
                    'LiftAndCoastDist': avg_lac,
                    'AvgBrakeIntensity': avg_brake_intensity
                })
                
                processed += 1
                
            except Exception as e:
                skipped += 1
                continue
        
        return {
            'driver': driver,
            'features': features,
            'processed': processed,
            'skipped': skipped
        }
        
    except Exception as e:
        return {
            'driver': driver,
            'features': [],
            'processed': 0,
            'skipped': len(race_laps_df[race_laps_df['Driver'] == driver])
        }


class MakeDataSet():
    def __init__(self, tyres_for_each_race, circuit_data, current_datetime, csv_path, pit, base_pace, test):
        self.tyres_for_each_race = tyres_for_each_race
        self.circuit_data = circuit_data
        self.current_datetime = current_datetime
        self.csv_path = csv_path
        self.schedule = fastf1.get_event_schedule(current_datetime.year) #pd dataframe
        self.past_races = self.schedule[self.schedule['Session5DateUtc'] < current_datetime]
        self.All_Laps = pd.DataFrame()
        self.pit = pit
        self.base_pace = base_pace
        self.test = test

    def hardness_mapping(self, a, b, c):
        hardness_map = {
                    'HARD': a,
                    'MEDIUM': b,
                    'SOFT': c,
                    'INTERMEDIATE':  7,
                    'WET':  8,
                }
        return hardness_map

    def encode_tyre_compound(self, compound, hardness_map):
        self.All_Laps[compound] = self.All_Laps['Compound'].map(hardness_map)
        self.All_Laps[compound] = self.All_Laps[compound].ffill().bfill()

    def merge_weather(self, weather):
            weather['Minute'] = pd.to_timedelta(weather['Time']).dt.components['minutes']
            hourly_weather = weather[((weather['Minute'].isin([0])))].drop('Minute', axis=1)
            hourly_weather['TimeStamp'] = pd.to_timedelta(hourly_weather['Time'])

            self.All_Laps['TimeStamp'] = pd.to_timedelta(self.All_Laps['Time']) 
            
            self.All_Laps['TimeBin'] = self.All_Laps['TimeStamp'].dt.floor('h')  # Rounds down to the hour

            self.All_Laps = pd.merge_asof(
                self.All_Laps.sort_values('TimeStamp'),
                hourly_weather.sort_values('TimeStamp'),
                left_on='TimeStamp',
                right_on='Time',
                direction='backward',  # Assigns weather from the most recent hour
                tolerance=pd.Timedelta('60min')  # Only match within the same hour
            )

    def merge_circuit(self, location):
        # Map FastF1 location names to circuit_data names
        mapped_location = location_mapping.get(location, location)
        
        matching_circuits = circuit_data[circuit_data['Name'] == mapped_location]
        if matching_circuits.empty:
            raise ValueError(f"Location '{mapped_location}' (original: '{location}') not found in circuit_data.")
        circuit_data_index = matching_circuits.index[0]
        circuit_info = circuit_data.iloc[circuit_data_index]

        circuit_info = circuit_info.to_frame().T
        # Prefix circuit columns to avoid name collisions and ensure they persist
        circuit_info.columns = [f"Circuit_{c.replace(' ', '_')}" for c in circuit_info.columns]

        duplicated_rows = pd.concat([circuit_info]*len(self.All_Laps), ignore_index=True)

        self.All_Laps = pd.concat([self.All_Laps, duplicated_rows], axis=1)
        # Debug: report which circuit columns were added
        added = [c for c in self.All_Laps.columns if c.startswith('Circuit_')]
        print(f"Added circuit columns for {mapped_location}: {added}")

    def fbfill_nas(self, col):
        self.All_Laps[col] = self.All_Laps[col].ffill().bfill()
        print(f"{self.All_Laps[col].isna().sum()} missing values in {col} after forward/backward fill")

    def total(self):
        self.All_Laps['TotalTime'] = self.All_Laps.groupby('Driver')['LapTime_sec'].cumsum() #cumulative sum of laptimes
        # Position is already provided by FastF1 - no need to calculate it
        # FastF1's Position column reflects the actual race position at each lap

    def gap_to_lead(self):
        leader_times = self.All_Laps[self.All_Laps['Position'] == 1].set_index('LapNumber')['TotalTime']
        self.All_Laps['LeaderTime'] = self.All_Laps['LapNumber'].map(leader_times)
        self.All_Laps['GapToLeader'] = self.All_Laps['TotalTime'] - self.All_Laps['LeaderTime'] #gap to leader

    def gap_to_ahead(self):
        """
        For each lap, sort drivers by TotalTime (ascending) and take the previous
        row as the 'driver ahead' -> robust to unsorted index or missing rows.
        """
        df = self.All_Laps
        # Rank drivers by time to ensure deterministic ordering
        df['RankByTime'] = df.groupby('LapNumber')['TotalTime'].rank(method='first', ascending=True)
        sorted_df = df.sort_values(['LapNumber', 'RankByTime'])
        sorted_df['NextDriverTime'] = sorted_df.groupby('LapNumber')['TotalTime'].shift(1)
        # Assign back using original index alignment
        self.All_Laps['NextDriverTime'] = sorted_df['NextDriverTime']
        self.All_Laps['GapToAhead'] = self.All_Laps['TotalTime'] - self.All_Laps['NextDriverTime']
        # Leader has no ahead
        self.All_Laps.loc[self.All_Laps.groupby('LapNumber')['RankByTime'].transform('min') == self.All_Laps['RankByTime'], 'GapToAhead'] = 0
        self.All_Laps.drop(columns=['RankByTime'], inplace=True, errors='ignore')

    def gap_to_behind(self):
        """
        Symmetric of gap_to_ahead: sort by TotalTime then look at the next row for the 'driver behind'.
        """
        df = self.All_Laps
        df['RankByTime'] = df.groupby('LapNumber')['TotalTime'].rank(method='first', ascending=True)
        sorted_df = df.sort_values(['LapNumber', 'RankByTime'])
        sorted_df['PrevDriverTime'] = sorted_df.groupby('LapNumber')['TotalTime'].shift(-1)
        self.All_Laps['PrevDriverTime'] = sorted_df['PrevDriverTime']
        self.All_Laps['GapToBehind'] = self.All_Laps['PrevDriverTime'] - self.All_Laps['TotalTime']
        # Last-placed driver has no behind
        max_rank = self.All_Laps.groupby('LapNumber')['RankByTime'].transform('max')
        self.All_Laps.loc[self.All_Laps['RankByTime'] == max_rank, 'GapToBehind'] = 0
        self.All_Laps.drop(columns=['RankByTime'], inplace=True, errors='ignore')



    def handle_last_place(self):
        self.All_Laps.loc[self.All_Laps['Position'] == self.All_Laps.groupby('LapNumber')['Position'].transform('max'), 'GapToBehind'] = 0
        self.All_Laps.loc[self.All_Laps['Position'] == 1, 'NextDriverTime'] = 0
        self.All_Laps.loc[self.All_Laps['Position'] == 1, 'GapToAhead'] = 0

    def tyre_life(self):
        """
        Calculate TyreLife per stint using FreshTyre flag.
        TyreLife resets to 0 when FreshTyre=True (new stint), then increments.
        """
        self.All_Laps = self.All_Laps.sort_values(['Driver', 'LapNumber'])
        
        # Create stint IDs: cumulative sum of FreshTyre events per driver
        self.All_Laps['StintID'] = self.All_Laps.groupby('Driver')['FreshTyre'].cumsum()
        
        # Within each stint, count laps from 0
        # Use fillna(0) to handle any missing FreshTyre values
        self.All_Laps['TyreLife'] = self.All_Laps.groupby(['Driver', 'StintID']).cumcount()
        
        # Drop temporary StintID column
        self.All_Laps.drop(columns=['StintID'], inplace=True, errors='ignore')
        
        print(f"TyreLife calculated: range {self.All_Laps['TyreLife'].min():.0f} to {self.All_Laps['TyreLife'].max():.0f}")
    def one_hot_track_status(self):
        one_hot = pd.get_dummies(
        self.All_Laps['TrackStatus'], 
            prefix='status'
        )
        # Remove any existing status columns to avoid duplicates
        status_cols = [col for col in self.All_Laps.columns if col.startswith('status_')]
        self.All_Laps = self.All_Laps.drop(columns=status_cols)
        # Add the new one-hot encoded status columns
        self.All_Laps = pd.concat([self.All_Laps, one_hot.astype(int)], axis=1)

    def time_since_last_weather(self):
        self.All_Laps["TimeSinceLastWeatherMeasurement"] = pd.to_timedelta(self.All_Laps["Time_x"]) - pd.to_timedelta(self.All_Laps["TimeStamp_y"])

    def csv(self):
        self.All_Laps.to_csv(self.csv_path, index=False)

# Get unique race names and shuffle
    def train_val_test(self):
        All_Laps = pd.read_csv(self.csv_path)
        all_races = All_Laps['Name'].unique()
        np.random.shuffle(all_races)  # Randomize to avoid season bias

        # Split ratios (adjust as needed)

        train_races, val_races, test_races = np.split(
            all_races, 
            [int(0.7 * len(all_races)), int(0.85 * len(all_races))]
        )

        # Create splits

        train = All_Laps[All_Laps['Name'].isin(train_races)]
        train.to_csv('All_Laps_Train.csv', index=False)
        unique_values = train['Name'].unique()
        print(unique_values)

        val = All_Laps[All_Laps['Name'].isin(val_races)]
        unique_values = val['Name'].unique()
        print(unique_values)
        val.to_csv('All_Laps_val.csv', index=False)

        test = All_Laps[All_Laps['Name'].isin(test_races)]
        test.to_csv('All_Laps_test.csv', index=False)
        unique_values = test['Name'].unique()
        print(unique_values)
    def driver_and_teams_to_int(self):
        # Use global mappings (created once in create_dataset)
        if not hasattr(self, 'driver_to_idx') or not hasattr(self, 'team_to_idx'):
            raise ValueError("Global driver/team mappings not initialized. Call create_dataset first.")
        
        # Convert to indices using global mappings
        self.All_Laps['Driver_idx'] = self.All_Laps['Driver'].map(self.driver_to_idx)
        self.All_Laps['Team_idx'] = self.All_Laps['Team'].map(self.team_to_idx)
        
        # Check for unmapped drivers/teams (shouldn't happen in training, might in test)
        unmapped_drivers = self.All_Laps['Driver_idx'].isna().sum()
        unmapped_teams = self.All_Laps['Team_idx'].isna().sum()
        
        if unmapped_drivers > 0:
            print(f"  ⚠ Warning: {unmapped_drivers} laps have drivers not in the mapping")
            # Fill with a special 'unknown' index (max_idx + 1)
            self.All_Laps['Driver_idx'] = self.All_Laps['Driver_idx'].fillna(len(self.driver_to_idx))
        
        if unmapped_teams > 0:
            print(f"  ⚠ Warning: {unmapped_teams} laps have teams not in the mapping")
            self.All_Laps['Team_idx'] = self.All_Laps['Team_idx'].fillna(len(self.team_to_idx))

    def clean_data_for_base_pace(self):
        self.All_Laps = self.All_Laps[self.All_Laps['GapToAhead'] >= 2]
        self.All_Laps = self.All_Laps[self.All_Laps['GapToBehind'] >= 2]
        self.All_Laps = self.All_Laps[self.All_Laps['dnf'] == 0]
        self.All_Laps = self.All_Laps[self.All_Laps['PitDuration'] == 0.0]
        # Only filter by status_4 if the column exists
        if 'status_1' in self.All_Laps.columns:
            self.All_Laps = self.All_Laps[self.All_Laps['status_1'] == 1]
        self.All_Laps = self.All_Laps.drop(['PitDuration', 'SpeedFL', 'SpeedST'], axis=1)

    def no_pits(self):
        # Filter for laps where there were no pit stops (NaT values)
        self.All_Laps = self.All_Laps[self.All_Laps['PitInTime'].isna()]
    
    def have_pits(self):
        # Filter for laps where there were pit stops (not NaT values)
        self.All_Laps = self.All_Laps[self.All_Laps['PitInTime'].notna()]

    def DNF(self):
        """
        Mark dnf only on the final lap row for drivers whose last lap is less
        than the race's maximum lap. (Previously this flagged all laps for that driver.)
        """
        race_max_lap = self.All_Laps['LapNumber'].max()
        driver_last_lap = self.All_Laps.groupby('Driver')['LapNumber'].transform('max')
        dnf_last_lap_mask = (self.All_Laps['LapNumber'] == driver_last_lap) & (driver_last_lap < race_max_lap)
        self.All_Laps['dnf'] = dnf_last_lap_mask.astype(int)

    def fill_laptimes_with_pit_duration(self):
        """
        Fill missing lap times by adding pit duration to normal lap time
        """
        pit_lap_mask = self.All_Laps['LapTime_sec'].isna() & self.All_Laps['PitDuration'].notna() & (self.All_Laps['PitDuration'] > 0)
        self.All_Laps.loc[pit_lap_mask, 'LapTime_sec'] = self.All_Laps.loc[pit_lap_mask, 'LapTime_sec'].ffill().bfill() 

    def drop_cols(self):
        # Columns to drop after all processing
        cols_to_drop = [
            # Time/timestamp columns (converted to numeric or not needed)
            'Time_x', 'LapStartTime', 'TimeStamp_x', 'TimeStamp_y', 'TimeBin', 'Time_y',
            
            # Categorical identifiers (use embeddings instead)
            'Driver', 'Team',
            
            # Pit stop columns (converted to numeric or intermediate calculations)
            'PitInTime', 'PitOutTime', 'NextPitOutTime', 'FreshTyre',
            
            # Track status (one-hot encoded)
            'TrackStatus',
            
            # Sector times (not needed for lap time prediction)
            'Sector1Time', 'Sector2Time', 'Sector3Time', 
            'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
            
            # FastF1 metadata columns
            'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate',
            
            # Intermediate calculation columns
            'TotalTime', 'LeaderTime', 'NextDriverTime', 'PrevDriverTime',
            
            # Speed columns 
            'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST',
            
            # Index columns
            'Unnamed: 0', 'Unnamed: 0.1',
            
            # Stint info (redundant with TyreLife)
            'Stint', 'DriverNumber',
            
            # Lap metadata
            'IsPersonalBest', 'LapStartDate',
            
            # Original lap time (keeping LapTime_sec)
            'LapTime'
            'status_14', 'status_126', 'status_16', 'status_167', 'status_67', 
            'status_671', 'status_71', 'status_26', 'status_6', 'status_2', 
            'status_6712', 'status_712'
        ]

        if not self.pit:
            cols_to_drop.append('PitDuration')
        
        # Only drop columns that actually exist in the dataframe
        existing_cols_to_drop = [col for col in cols_to_drop if col in self.All_Laps.columns]
        self.All_Laps = self.All_Laps.drop(existing_cols_to_drop, axis=1)
    def pit_duration(self):
        # Calculate pit durations without filtering the data
        self.All_Laps['NextPitOutTime'] = self.All_Laps.groupby('Driver')['PitOutTime'].shift(-1)
        # Compute duration only where PitInTime is not null
        self.All_Laps['PitDuration'] = (
            (self.All_Laps['NextPitOutTime'] - self.All_Laps['PitInTime'])
        ).fillna(0)
    def convert_to_numbers(self):
        self.All_Laps.loc[:, 'TimeSinceLastWeatherMeasurement'] = pd.to_timedelta(self.All_Laps['TimeSinceLastWeatherMeasurement']).dt.total_seconds()
        self.All_Laps['Rainfall'] = (
            self.All_Laps['Rainfall']
            .fillna(False)
            .astype(int)
        )
        self.All_Laps.loc[:, 'dnf'] = self.All_Laps['dnf'].astype(int)

    def extract_telemetry_features(self):
        """
        SERIAL telemetry extraction (no multiprocessing - FastF1 Lap objects aren't picklable).
        - Throttle commitment: % time at WOT (>95%) in acceleration zones
        - Lift-and-coast: Average distance between throttle close and brake application
        - Brake intensity: Average deceleration rate (proxy for brake pressure)
        """
        import numpy as np
        
        print(f"Extracting telemetry features (serial mode)...")
        
        # Initialize feature columns with defaults
        self.All_Laps['ThrottleCommitment'] = 0.0
        self.All_Laps['LiftAndCoastDist'] = 0.0
        self.All_Laps['AvgBrakeIntensity'] = 0.0
        
        # Get unique drivers in this race
        unique_drivers = self.All_Laps['Driver'].unique()
        total_drivers = len(unique_drivers)
        
        print(f"  Processing {total_drivers} drivers...")
        
        total_processed = 0
        total_skipped = 0
        
        # Process each driver serially
        for idx, driver in enumerate(unique_drivers, 1):
            driver_laps = self.All_Laps[self.All_Laps['Driver'] == driver]
            
            for _, lap in driver_laps.iterrows():
                lap_num = lap['LapNumber']
                try:
                    # Get telemetry for this specific lap
                    lap_telemetry = lap.get_telemetry()
                    
                    if lap_telemetry is None or len(lap_telemetry) < 10:
                        total_skipped += 1
                        continue
                    
                    # Extract arrays
                    throttle = lap_telemetry['Throttle'].values
                    brake = lap_telemetry['Brake'].values
                    speed = lap_telemetry['Speed'].values
                    distance = lap_telemetry['Distance'].values
                    
                    # 1. THROTTLE COMMITMENT
                    wot_mask = throttle > 95
                    throttle_commitment = wot_mask.sum() / len(throttle) if len(throttle) > 0 else 0
                    
                    # 2. LIFT-AND-COAST
                    brake_on = np.where(np.diff(brake.astype(int)) > 0)[0]
                    lac_distances = []
                    
                    for brake_idx in brake_on:
                        if brake_idx < 5:
                            continue
                        
                        throttle_before = throttle[max(0, brake_idx-50):brake_idx]
                        throttle_close_points = np.where(throttle_before < 5)[0]
                        
                        if len(throttle_close_points) > 0:
                            last_throttle_close_idx = throttle_close_points[-1] + max(0, brake_idx-50)
                            
                            if last_throttle_close_idx < brake_idx and brake_idx < len(distance):
                                coast_dist = distance[brake_idx] - distance[last_throttle_close_idx]
                                if coast_dist > 0:
                                    lac_distances.append(coast_dist)
                    
                    avg_lac = np.mean(lac_distances) if len(lac_distances) > 0 else 0
                    
                    # 3. BRAKE INTENSITY
                    brake_mask = brake > 0
                    
                    if brake_mask.sum() > 1:
                        speed_change = -np.diff(speed)
                        speed_change_braking = speed_change[brake_mask[:-1]]
                        avg_brake_intensity = np.mean(speed_change_braking) if len(speed_change_braking) > 0 else 0
                    else:
                        avg_brake_intensity = 0
                    
                    # Update DataFrame
                    mask = (self.All_Laps['Driver'] == driver) & (self.All_Laps['LapNumber'] == lap_num)
                    self.All_Laps.loc[mask, 'ThrottleCommitment'] = throttle_commitment
                    self.All_Laps.loc[mask, 'LiftAndCoastDist'] = avg_lac
                    self.All_Laps.loc[mask, 'AvgBrakeIntensity'] = avg_brake_intensity
                    
                    total_processed += 1
                    
                except Exception as e:
                    total_skipped += 1
                    continue
            
            print(f"  ✓ [{idx}/{total_drivers}] {driver}: completed")
        
        print(f"\n  ✓ COMPLETE: Processed {total_processed} laps, skipped {total_skipped} laps")
        
        # Print statistics
        print(f"  ThrottleCommitment: mean={self.All_Laps['ThrottleCommitment'].mean():.3f}, std={self.All_Laps['ThrottleCommitment'].std():.3f}")
        print(f"  LiftAndCoastDist: mean={self.All_Laps['LiftAndCoastDist'].mean():.1f}m, std={self.All_Laps['LiftAndCoastDist'].std():.1f}m")
        print(f"  AvgBrakeIntensity: mean={self.All_Laps['AvgBrakeIntensity'].mean():.2f}, std={self.All_Laps['AvgBrakeIntensity'].std():.2f}")

    def fit_scalers(self):
        """
        Fits scalers on the current data (TRAINING DATA ONLY).
        - Gaussian-like: StandardScaler
        - Skewed: Log1p + StandardScaler
        - Sequential: Min-Max
        Call this ONLY on training data, then save scalers.
        """
        
        # 1. Define Column Groups
        # Gaussian-like continuous features (skewness < 1.0)
        # IMPORTANT: LapTime_sec is the TARGET and must be here for simple z-score normalization
        gaussian_cols = [
            'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindDirection', 
            'TimeSinceLastWeatherMeasurement', 'CircuitLength', 
            'Number of Laps', 'Circuit_CircuitLength', 'Circuit_Number_of_Laps',
            'Circuit_NumberOfTurns', 'Circuit_AverageAngleAbs', 'Circuit_AverageAngle',
            'ThrottleCommitment', 'LiftAndCoastDist', 'AvgBrakeIntensity',
            'LapTime_sec'  # TARGET VARIABLE - simple z-score normalization
        ]
        
        # Skewed features (skewness > 1.0) - apply log1p transform then standardize
        # NOTE: LapTime_sec is the TARGET - it should be z-score normalized, NOT log-transformed
        skewed_cols = ['GapToLeader', 'GapToAhead', 'GapToBehind', 'WindSpeed', 'Prev_LapTime']
        
        # Sequential/Counter features
        sequential_cols = ['LapNumber', 'TyreLife', 'Position']
        
        # 2. Pre-processing: Save stats and fill NaNs
        if 'LapTime_sec' in self.All_Laps.columns:
            self.lap_mean = self.All_Laps['LapTime_sec'].mean()
            self.lap_std = self.All_Laps['LapTime_sec'].std()
        
        self.scalers = {}
        self.All_Laps = self.All_Laps.fillna(0)

        # 3. Fit and Transform Training Data
        
        # A. Skewed features: Log Transform then fit StandardScaler
        skewed_scalers = {}
        for col in skewed_cols:
            if col in self.All_Laps.columns:
                self.All_Laps[col] = np.log1p(self.All_Laps[col])
                ss_skew = StandardScaler()
                self.All_Laps[[col]] = ss_skew.fit_transform(self.All_Laps[[col]])
                skewed_scalers[col] = ss_skew
        if skewed_scalers:
            self.scalers['skewed'] = skewed_scalers

        # B. Gaussian features: Fit StandardScaler
        gaussian_to_process = [c for c in gaussian_cols if c in self.All_Laps.columns]
        if gaussian_to_process:
            ss_gauss = StandardScaler()
            self.All_Laps[gaussian_to_process] = ss_gauss.fit_transform(self.All_Laps[gaussian_to_process])
            self.scalers['gaussian'] = ss_gauss

        # C. Sequential features: Fit MinMaxScaler
        sequential_to_process = [c for c in sequential_cols if c in self.All_Laps.columns]
        if sequential_to_process:
            mm_scaler = MinMaxScaler()
            self.All_Laps[sequential_to_process] = mm_scaler.fit_transform(self.All_Laps[sequential_to_process])
            self.scalers['minmax'] = mm_scaler
        
        # 4. Save scalers AND driver/team mappings to disk
        try:
            if hasattr(self, 'lap_mean') and hasattr(self, 'lap_std'):
                self.scalers['lap_mean'] = float(self.lap_mean)
                self.scalers['lap_std'] = float(self.lap_std)
            
            # Save driver and team mappings (CRITICAL for consistency)
            if hasattr(self, 'driver_to_idx') and hasattr(self, 'team_to_idx'):
                self.scalers['driver_to_idx'] = self.driver_to_idx
                self.scalers['team_to_idx'] = self.team_to_idx
                print(f"  ✓ Saved {len(self.driver_to_idx)} driver mappings, {len(self.team_to_idx)} team mappings")

            scaler_path = os.path.splitext(self.csv_path)[0] + '_scalers.joblib'
            joblib.dump(self.scalers, scaler_path)
            print(f"✓ Fitted scalers on training data and saved to {scaler_path}")
        except Exception as e:
            print(f"✗ Failed to save scalers: {e}")

    def apply_scalers(self, scaler_path):
        """
        Applies pre-fitted scalers to transform the current data.
        Use this for validation and test data (transform only, no fitting).
        """
        # Load saved scalers
        if not os.path.exists(scaler_path):
            raise FileNotFoundError(f"Scaler file not found: {scaler_path}")
        
        scalers = joblib.load(scaler_path)
        print(f"✓ Loaded scalers from {scaler_path}")
        
        # Load driver and team mappings (should already be set in create_dataset, but double-check)
        if 'driver_to_idx' in scalers and 'team_to_idx' in scalers:
            self.driver_to_idx = scalers['driver_to_idx']
            self.team_to_idx = scalers['team_to_idx']
            print(f"  ✓ Loaded {len(self.driver_to_idx)} driver mappings, {len(self.team_to_idx)} team mappings")
        
        # Define column groups (MUST MATCH fit_scalers exactly)
        gaussian_cols = [
            'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindDirection', 
            'TimeSinceLastWeatherMeasurement', 'CircuitLength', 
            'Number of Laps', 'Circuit_CircuitLength', 'Circuit_Number_of_Laps',
            'Circuit_NumberOfTurns', 'Circuit_AverageAngleAbs', 'Circuit_AverageAngle',
            'ThrottleCommitment', 'LiftAndCoastDist', 'AvgBrakeIntensity',
            'LapTime_sec'  # TARGET VARIABLE - simple z-score normalization
        ]
        skewed_cols = ['GapToLeader', 'GapToAhead', 'GapToBehind', 'WindSpeed', 'Prev_LapTime']
        sequential_cols = ['LapNumber', 'TyreLife', 'Position']
        
        # Fill NaNs
        self.All_Laps = self.All_Laps.fillna(0)

        # Apply transformations using pre-fitted scalers
        
        # A. Skewed features: Log transform then apply StandardScaler
        if 'skewed' in scalers:
            for col in skewed_cols:
                if col in self.All_Laps.columns and col in scalers['skewed']:
                    self.All_Laps[col] = np.log1p(self.All_Laps[col])
                    ss_skew = scalers['skewed'][col]
                    self.All_Laps[[col]] = ss_skew.transform(self.All_Laps[[col]])

        # B. Gaussian features: Apply StandardScaler
        gaussian_to_process = [c for c in gaussian_cols if c in self.All_Laps.columns]
        if gaussian_to_process and 'gaussian' in scalers:
            ss_gauss = scalers['gaussian']
            self.All_Laps[gaussian_to_process] = ss_gauss.transform(self.All_Laps[gaussian_to_process])

        # C. Sequential features: Apply MinMaxScaler
        sequential_to_process = [c for c in sequential_cols if c in self.All_Laps.columns]
        if sequential_to_process and 'minmax' in scalers:
            mm_scaler = scalers['minmax']
            self.All_Laps[sequential_to_process] = mm_scaler.transform(self.All_Laps[sequential_to_process])
        
        print(f"✓ Applied scalers to {len(self.All_Laps)} samples")

    def subtract_pit_duration_from_laptime(self):
        self.All_Laps['LapTime_sec'] = self.All_Laps['LapTime_sec'] - self.All_Laps['PitDuration']
    
    def create_dataset(self, fit_scalers=True, scaler_path=None):
        all_races_data = []  # List to accumulate data from all races
        
        # Define validation circuits (ZERO OVERLAP - circuit-level split)
        # This tests generalization to completely unseen circuits
        validation_circuits = ['Silverstone', 'Monza', 'Montréal', 'Miami', 'Imola', 'Barcelona']
        
        # STEP 1: Create global driver and team mappings (MUST be done before processing races)
        if fit_scalers:
            # Training mode: Create mappings from ALL drivers/teams in the schedule
            print("Creating global driver and team mappings from schedule...")
            all_drivers = set()
            all_teams = set()
            
            for location in self.past_races['Location'].tolist():
                try:
                    race = fastf1.get_session(self.current_datetime.year, location, 'R')
                    race.load(weather=False, messages=False, laps=True, telemetry=False)
                    all_drivers.update(race.laps['Driver'].unique())
                    all_teams.update(race.laps['Team'].unique())
                except Exception as e:
                    print(f"  ⚠ Could not load {location} for mapping: {e}")
                    continue
            
            # Create consistent mappings sorted alphabetically for reproducibility
            self.driver_to_idx = {driver: idx for idx, driver in enumerate(sorted(all_drivers))}
            self.team_to_idx = {team: idx for idx, team in enumerate(sorted(all_teams))}
            
            print(f"  ✓ Created mappings: {len(self.driver_to_idx)} drivers, {len(self.team_to_idx)} teams")
        else:
            # Validation/Test mode: Load mappings from scaler file
            if scaler_path is None:
                raise ValueError("scaler_path must be provided when fit_scalers=False")
            
            scalers = joblib.load(scaler_path)
            if 'driver_to_idx' not in scalers or 'team_to_idx' not in scalers:
                raise ValueError(f"Driver/team mappings not found in {scaler_path}")
            
            self.driver_to_idx = scalers['driver_to_idx']
            self.team_to_idx = scalers['team_to_idx']
            print(f"  ✓ Loaded mappings: {len(self.driver_to_idx)} drivers, {len(self.team_to_idx)} teams")
        
        if self.test:
            # Validation dataset: ONLY the 6 test circuits
            locations = validation_circuits
            print(f"Validation: Using ONLY {validation_circuits} (zero overlap with training)")
        else:
            # Training dataset: ALL circuits EXCEPT the 6 validation circuits
            all_locations = self.past_races['Location'].tolist()
            # Map locations and filter out validation circuits
            locations = [
                loc for loc in all_locations 
                if location_mapping.get(loc, loc) not in validation_circuits
            ]
            print(f"Training on {len(locations)} race instances, EXCLUDING validation circuits: {validation_circuits}")
        
        for location in locations:
            # Map FastF1 location names to circuit_data names for tyre lookup
            mapped_location = location_mapping.get(location, location)
            
            hardness_map = None
            for key in self.tyres_for_each_race:
                if mapped_location in self.tyres_for_each_race[key]:
                    # Extract the three numbers from the key string
                    a, b, c = int(key[0]), int(key[1]), int(key[2])
                    hardness_map = self.hardness_mapping(a, b, c)
                    break
            
            if hardness_map is None:
                print(f"Warning: No tyre mapping found for location '{location}' (mapped: '{mapped_location}'), skipping this race.")
                continue  # Skip this race if no tyre mapping is found
            
            race = fastf1.get_session(self.current_datetime.year, location,'R' )
            race.load(weather=True, messages=False)
            Laps = race.laps
            weather = race.weather_data #need to drop  things that weather forecast APIs dont have (by the minute)
                
            self.All_Laps = Laps  # Start with the laps data
            self.current_race_laps = Laps  # Store for telemetry feature extraction
            
            self.merge_weather(weather)
            self.merge_circuit(location)
                    
            self.All_Laps = self.All_Laps.sort_values(['Driver', 'LapNumber'])

            self.encode_tyre_compound('Compound', hardness_map)
            
            self.All_Laps['LapTime_sec'] = self.All_Laps['LapTime'].dt.total_seconds()
            self.All_Laps['Prev_LapTime'] = self.All_Laps['LapTime_sec'].shift(1)
            self.pit_duration()
            self.All_Laps['PitDuration'] = pd.to_timedelta(self.All_Laps['PitDuration']).dt.total_seconds()
            self.fill_laptimes_with_pit_duration()

            self.fbfill_nas('LapTime_sec')
            
            self.All_Laps = self.All_Laps.sort_values(['Driver', 'LapNumber'])
            self.total()
            self.gap_to_lead()
            self.gap_to_ahead()
            self.gap_to_behind()
            self.handle_last_place()
            gap_cols = ['GapToLeader', 'GapToAhead', 'GapToBehind', 'Position']

            # Save lap 1 positions (qualifying grid positions) before shifting
            lap1_positions = self.All_Laps[self.All_Laps['LapNumber'] == 1][['Driver', 'Position']].copy()
            
            self.All_Laps[gap_cols] = (
                self.All_Laps
                .sort_values(['Driver', 'LapNumber'])
                .groupby('Driver')[gap_cols]
                .shift(1)
            )
            self.All_Laps[gap_cols] = self.All_Laps[gap_cols].fillna(0.0)
            
            # Restore lap 1 positions from qualifying (don't lag these - we know them)
            for _, row in lap1_positions.iterrows():
                mask = (self.All_Laps['Driver'] == row['Driver']) & (self.All_Laps['LapNumber'] == 1)
                self.All_Laps.loc[mask, 'Position'] = row['Position']
            
            # REMOVED: self.tyre_life() - FastF1 already provides correct TyreLife column
            # The FreshTyre flag is broken (True for almost all laps), so recalculating
            # TyreLife creates incorrect values (massive spike at 0). Use FastF1's built-in.
            
            self.one_hot_track_status()
            self.time_since_last_weather()
            
            # Extract telemetry features (throttle, LAC, brake)
            self.extract_telemetry_features()
            
            self.driver_and_teams_to_int()
            
            self.DNF()
            
            if self.pit:
                self.have_pits()
                self.subtract_pit_duration_from_laptime()
            else:   
                self.no_pits()
            if self.base_pace:
                self.clean_data_for_base_pace()
            
            self.convert_to_numbers()
            self.drop_cols()
            
            
            # Add processed race data to the list
            all_races_data.append(self.All_Laps.copy())
            print(f"Processed {location} - {len(self.All_Laps)} laps")
        
        # Combine all race data
        if all_races_data:
            self.All_Laps = pd.concat(all_races_data, ignore_index=True)
            
            # Apply normalization
            if fit_scalers:
                # Training mode: Fit scalers on this data
                print("\nFitting scalers on TRAINING data...")
                self.fit_scalers()
            else:
                # Validation/Test mode: Apply pre-fitted scalers
                if scaler_path is None:
                    raise ValueError("scaler_path must be provided when fit_scalers=False")
                print(f"\nApplying pre-fitted scalers from {scaler_path}...")
                self.apply_scalers(scaler_path)
            
            print(f"Total dataset created with {len(self.All_Laps)} laps from {len(all_races_data)} races")
        else:
            print("No race data was processed successfully")
            self.All_Laps = pd.DataFrame()
        
        return self.All_Laps

In [0]:
from pyspark.sql import SparkSession

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

# Use 2024 instead of current year since 2026 race data is not available yet
current_datetime = datetime(2025, 12, 31)

# STEP 1: Create TRAINING dataset and FIT scalers
print("="*60)
print("STEP 1: Creating TRAINING dataset and FITTING scalers...")
print("="*60)
data_processor_train = MakeDataSet(tyre_for_each_race, circuit_data, current_datetime, 'training_temp.csv', False, False, False)
training_laps = data_processor_train.create_dataset(fit_scalers=True)  # Fit scalers on training data

# Write training data to Unity Catalog
print("\nWriting training data to Unity Catalog...")
spark_train_df = spark.createDataFrame(training_laps)
spark_train_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("workspace.f1_racing_laptime_pred.raw_training")
print(f"✓ Saved {len(training_laps)} training laps to workspace.f1_racing_laptime_pred.raw_training")

# STEP 2: Create VALIDATION dataset and APPLY scalers (transform only)
print("\n" + "="*60)
print("STEP 2: Creating VALIDATION dataset and APPLYING scalers (transform only)...")
print("="*60)
data_processor_val = MakeDataSet(tyre_for_each_race, circuit_data, current_datetime, 'validation_temp.csv', False, False, True)
validation_laps = data_processor_val.create_dataset(fit_scalers=False, scaler_path='training_temp_scalers.joblib')  # Apply training scalers

# Write validation data to Unity Catalog
print("\nWriting validation data to Unity Catalog...")
spark_val_df = spark.createDataFrame(validation_laps)
spark_val_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("workspace.f1_racing_laptime_pred.raw_validating")
print(f"✓ Saved {len(validation_laps)} validation laps to workspace.f1_racing_laptime_pred.raw_validating")

print("\n" + "="*60)
print("COMPLETE! Data saved to Unity Catalog:")
print("  - workspace.f1_racing_laptime_pred.raw_training (with FITTED scalers)")
print("  - workspace.f1_racing_laptime_pred.raw_validating (with APPLIED scalers)")
print("  - Scalers saved to: training_temp_scalers.joblib")
print("="*60)

In [0]:
import json
import glob
import numpy as np
from pyproj import Transformer
from scipy.signal import savgol_filter, find_peaks

def preprocess_circuit(coords):
    """
    Convert lat/lon to UTM, resample to even arc-length spacing, smooth, and trim padding.
    """
    lons, lats = zip(*coords)
    utm_zone = int((np.mean(lons) + 180) / 6) + 1
    transformer = Transformer.from_crs("EPSG:4326", f"EPSG:326{utm_zone:02d}", always_xy=True)
    x, y = transformer.transform(lons, lats)
    x, y = np.array(x), np.array(y)

    # closed-loop wraparound padding
    pad = 15
    x_pad = np.concatenate([x[-pad:], x, x[:pad]])
    y_pad = np.concatenate([y[-pad:], y, y[:pad]])

    # resample to even arc-length spacing
    dist = np.concatenate([[0], np.cumsum(np.hypot(np.diff(x_pad), np.diff(y_pad)))])
    even_dist = np.arange(0, dist[-1], 5)  # every 5m
    x_even = np.interp(even_dist, dist, x_pad)
    y_even = np.interp(even_dist, dist, y_pad)

    x_smooth = savgol_filter(x_even, 15, 3)
    y_smooth = savgol_filter(y_even, 15, 3)

    # trim padding back off (in resampled index space)
    trim = int(pad * (len(even_dist) / len(x_pad)))
    return x_smooth[trim:-trim], y_smooth[trim:-trim]

def compute_curvature(x, y):
    """
    Compute signed curvature: +left, -right
    """
    dx, dy = np.gradient(x), np.gradient(y)
    ddx, ddy = np.gradient(dx), np.gradient(dy)
    return (dx * ddy - dy * ddx) / (dx**2 + dy**2)**1.5

def engineer_circuit_features(geojson):
    """
    Extract all geometric features from a circuit GeoJSON.
    """
    feat = geojson['features'][0]
    props = feat['properties']
    coords = feat['geometry']['coordinates']
    x, y = preprocess_circuit(coords)
    k = compute_curvature(x, y)

    seg_len = np.hypot(np.diff(x), np.diff(y))
    computed_length = seg_len.sum()

    peaks, _ = find_peaks(np.abs(k), height=1/300, distance=20)
    radii = 1 / np.abs(k[peaks]) if len(peaks) else np.array([])

    straight_mask = np.abs(k) < 1/500
    
    # Find longest straight
    straight_runs = np.diff(np.concatenate([[0], straight_mask.astype(int), [0]]))
    run_starts = np.where(straight_runs == 1)[0]
    run_ends = np.where(straight_runs == -1)[0]
    run_lengths = run_ends - run_starts
    longest_straight = run_lengths.max() * 5 if len(run_lengths) else 0  # 5m per point
    
    bbox = feat['bbox']
    bbox_w = bbox[2] - bbox[0]
    bbox_h = bbox[3] - bbox[1]
    bbox_area = bbox_w * bbox_h
    
    # Rotation-invariant elongation
    bbox_elongation = max(bbox_w, bbox_h) / min(bbox_w, bbox_h)
    
    # Left/right turn balance
    left_turns = (k[peaks] > 0).sum() if len(peaks) else 0
    right_turns = (k[peaks] < 0).sum() if len(peaks) else 0
    lr_ratio = left_turns / max(right_turns, 1)

    return {
        'Circuit_Name': props['Name'],
        'length_official': props['length'],
        'length_computed': computed_length,
        'altitude': props['altitude'],
        'num_corners': len(peaks),
        'mean_corner_radius': radii.mean() if len(radii) else np.nan,
        'min_corner_radius': radii.min() if len(radii) else np.nan,
        'std_corner_radius': radii.std() if len(radii) else np.nan,
        'straight_ratio': straight_mask.mean(),
        'longest_straight': longest_straight,
        'total_turning': (np.abs(k[:-1]) * seg_len).sum(),
        'left_right_ratio': lr_ratio,
        'bbox_elongation': bbox_elongation,
        'compactness': computed_length**2 / bbox_area,
    }

# Use f1-circuits repo from workspace
import os
circuit_repo_path = '/Workspace/Users/s3yuen@uwaterloo.ca/f1-circuits'
circuits_dir = os.path.join(circuit_repo_path, 'circuits')

if not os.path.exists(circuits_dir):
    raise FileNotFoundError(f"Could not find circuits directory at {circuits_dir}")

print(f"✓ Using f1-circuits repo at: {circuit_repo_path}")

# Find all circuit GeoJSON files
circuit_files = glob.glob(os.path.join(circuit_repo_path, 'circuits/*.geojson'))
print(f"Found {len(circuit_files)} circuit files")

# Extract features for each circuit (use newest layout)
circuit_features = []
for file_path in sorted(circuit_files):
    try:
        with open(file_path, 'r') as f:
            geojson = json.load(f)
        features = engineer_circuit_features(geojson)
        circuit_features.append(features)
        print(f"✓ {features['Circuit_Name']}: {features['num_corners']} corners, {features['computed_length']:.0f}m")
    except Exception as e:
        print(f"✗ Failed to process {file_path}: {e}")

# Create lookup table
circuit_geom_features = pd.DataFrame(circuit_features)

# Map circuit names to match FastF1 location names
circuit_name_mapping = {
    'Montréal': 'Montréal',
    'São Paulo': 'São Paulo',
    'Miami': 'Miami',
    # Add more as needed based on what's in the f1-circuits repo
}

for old_name, new_name in circuit_name_mapping.items():
    circuit_geom_features.loc[circuit_geom_features['Circuit_Name'] == old_name, 'Circuit_Name'] = new_name

print(f"\n{'='*60}")
print("Circuit Geometric Features Extracted:")
print(f"{'='*60}")
display(circuit_geom_features)

# Save for merging with training/testing data
circuit_geom_features.to_csv('circuit_geometric_features.csv', index=False)
print(f"\n✓ Saved circuit geometric features to circuit_geometric_features.csv")


# Circuit Geometric Feature Equations

## Preprocessing Pipeline

### 1. Coordinate Transformation
Convert lat/lon to UTM (Universal Transverse Mercator) projection:
```
UTM_zone = floor((mean_longitude + 180) / 6) + 1
(x, y) = Transform(lat, lon)  using EPSG:4326 → EPSG:326XX
```

### 2. Closed-Loop Padding
For smooth derivatives at start/finish line:
```
x_padded = [x[-15:], x, x[:15]]
y_padded = [y[-15:], y, y[:15]]
```

### 3. Arc-Length Resampling
Resample to evenly-spaced points every 5m:
```
dist[i] = Σ sqrt((x[i] - x[i-1])² + (y[i] - y[i-1])²)  for i=1..N
even_dist = [0, 5, 10, 15, ..., max(dist)]
x_even = interpolate(even_dist, dist, x_padded)
y_even = interpolate(even_dist, dist, y_padded)
```

### 4. Savitzky-Golay Smoothing
Smooth with window=15, polynomial order=3:
```
x_smooth = savgol_filter(x_even, 15, 3)
y_smooth = savgol_filter(y_even, 15, 3)
```

### 5. Trim Padding
Remove the padded sections from resampled array

---

## Curvature Computation

**Signed curvature** κ (kappa) at each point:

```
κ(s) = (x'(s) · y''(s) - y'(s) · x''(s)) / (x'(s)² + y'(s)²)^(3/2)
```

where:
- `x'(s), y'(s)` = first derivatives (computed via `np.gradient`)
- `x''(s), y''(s)` = second derivatives
- **Sign convention**: κ > 0 → left turn, κ < 0 → right turn
- **Magnitude**: |κ| = 1/radius (high curvature = tight corner)

---

## Extracted Features

### 1. **length_official** (m)
```
length_official = properties['length']  # From GeoJSON metadata
```
The circuit's official quoted length.

---

### 2. **length_computed** (m)
```
seg_len[i] = sqrt((x[i+1] - x[i])² + (y[i+1] - y[i])²)  for i=0..N-1
length_computed = Σ seg_len[i]
```
Actual path length computed from preprocessed coordinates.

**Cross-check**: Large divergence from `length_official` flags stale/incomplete geometry.

---

### 3. **altitude** (m)
```
altitude = properties['altitude']  # From GeoJSON metadata
```
Single elevation value — air density proxy (not an elevation profile).

---

### 4. **num_corners**
```
peaks = find_peaks(|κ|, height=1/300, distance=20)
num_corners = len(peaks)
```
**Parameters**:
- `height=1/300` → minimum curvature threshold (radius < 300m)
- `distance=20` → minimum separation between corners (20 × 5m = 100m)

---

### 5. **mean_corner_radius** (m)
```
radius[i] = 1 / |κ[peak_i]|  for each detected corner
mean_corner_radius = mean(radius)
```
Average turning radius across all detected corners.

---

### 6. **min_corner_radius** (m)
```
min_corner_radius = min(1 / |κ[peak_i]|)  for all peaks
```
Tightest corner on the circuit (smallest radius = highest curvature).

---

### 7. **std_corner_radius** (m)
```
std_corner_radius = std(1 / |κ[peak_i]|)  for all peaks
```
Variability in corner tightness. High std → mixed slow/fast corners (Monaco). Low std → consistent corner speeds (Silverstone).

---

### 8. **straight_ratio**
```
straight_mask = (|κ| < 1/500)  # radius > 500m
straight_ratio = (Σ straight_mask) / N
```
Fraction of the lap where curvature is negligible (essentially straight). Range: [0, 1].

---

### 9. **longest_straight** (m)
```
runs = contiguous_runs_where(|κ| < 1/500)
longest_straight = max(run_length) × 5m
```
Length of the longest straight section. Important for top speed and DRS.

---

### 10. **total_turning** (radians)
```
total_turning = Σ (|κ[i]| × seg_len[i])  for i=0..N-1
```
Integral of absolute curvature over the entire lap. Measures overall "twistiness":
- **High**: Monaco, Hungary (many direction changes)
- **Low**: Monza (flowing, few turns)

Independent of discrete corner count.

---

### 11. **left_right_ratio**
```
left_turns = count(κ[peaks] > 0)
right_turns = count(κ[peaks] < 0)
left_right_ratio = left_turns / max(right_turns, 1)
```
Balance between left and right corners:
- **≈ 1.0**: Balanced (equal left/right)
- **> 1.0**: Left-dominated
- **< 1.0**: Right-dominated

Relevant for asymmetric tire wear.

---

### 12. **bbox_elongation** (rotation-invariant)
```
bbox_w = longitude_max - longitude_min
bbox_h = latitude_max - latitude_min
bbox_elongation = max(bbox_w, bbox_h) / min(bbox_w, bbox_h)
```
Shape of circuit footprint:
- **≈ 1.0**: Square/compact layout (Monaco, Hungaroring)
- **> 2.0**: Highly elongated (Spa, Red Bull Ring, Baku)
- **1.0–2.0**: Moderately elongated

**Note**: Uses `max/min` instead of `w/h` to ensure rotation invariance.

---

### 13. **compactness**
```
compactness = length_computed² / bbox_area
```
How much track is crammed into the footprint:
- **High**: Technical, tight circuits (Monaco) — lots of track in small area
- **Low**: Flowing, spread-out circuits (Spa) — large footprint relative to length

Dimensionless measure of layout density.

---

## Tuning Guidelines

### Corner Detection Thresholds
Adjust in `find_peaks(|κ|, height=..., distance=...)`:

* **height**: Minimum curvature to count as a corner
  - Current: `1/300` → radius < 300m
  - Increase to detect only tight corners
  - Decrease to include gentle bends

* **distance**: Minimum separation between corners (in points)
  - Current: `20` → 100m minimum separation
  - Increase to merge chicanes into single corners
  - Decrease to separate complex sections

### Validation
Plot `|κ(s)|` vs distance for a known track (Monaco or Monza) and verify detected peaks align with actual corners before trusting the counts across all 24 circuits.

In [0]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

# Load circuit geometric features
if 'circuit_geom_features' not in dir():
    circuit_geom_features = pd.read_csv('circuit_geometric_features.csv')
    print(f"✓ Loaded {len(circuit_geom_features)} circuits from CSV")
else:
    print(f"✓ Using circuit_geom_features from previous cell ({len(circuit_geom_features)} circuits)")

# Display the features we're adding
print("\nCircuit geometric features to merge:")
print(circuit_geom_features.columns.tolist())

# Map dataset location names to CSV full circuit names
# Dataset uses short names (e.g., 'Spielberg'), CSV uses full names (e.g., 'Red Bull Ring')
location_to_circuit_name = {
    'Sakhir': 'Bahrain International Circuit',
    'Jeddah': 'Jeddah Corniche Circuit', 
    'Melbourne': 'Albert Park Circuit',
    'Suzuka': 'Suzuka International Racing Course',
    'Shanghai': 'Shanghai International Circuit',
    'Miami': 'Miami International Autodrome',
    'Imola': 'Autodromo Enzo e Dino Ferrari',
    'Monaco': 'Circuit de Monaco',
    'Montréal': 'Circuit Gilles-Villeneuve',
    'Barcelona': 'Circuit de Barcelona-Catalunya',
    'Spielberg': 'Red Bull Ring',
    'Silverstone': 'Silverstone Circuit',
    'Budapest': 'Hungaroring',
    'Spa-Francorchamps': 'Circuit de Spa-Francorchamps',
    'Zandvoort': 'Circuit Zandvoort',
    'Monza': 'Autodromo Nazionale Monza',
    'Baku': 'Baku City Circuit',
    'Marina Bay': 'Marina Bay Street Circuit',
    'Austin': 'Circuit of the Americas',
    'Mexico City': 'Autódromo Hermanos Rodríguez',
    'São Paulo': 'Autódromo José Carlos Pace - Interlagos',
    'Las Vegas': 'Las Vegas Street Circuit',
    'Lusail': 'Losail International Circuit',
    'Yas Island': 'Yas Marina Circuit',
}

# Create lookup dictionary (dataset location -> circuit geometric features)
circuit_geom_lookup = {}
for _, row in circuit_geom_features.iterrows():
    csv_circuit_name = row['Circuit_Name']
    # Find which dataset location maps to this CSV circuit name
    for dataset_location, csv_name in location_to_circuit_name.items():
        if csv_name == csv_circuit_name:
            circuit_geom_lookup[dataset_location] = row.to_dict()
            break

print(f"\nCreated lookup for {len(circuit_geom_lookup)} circuit locations")

# Load training and validation datasets from Unity Catalog
print("\n" + "="*60)
print("Loading datasets from Unity Catalog...")
print("="*60)

training_df = spark.table("workspace.f1_racing_laptime_pred.raw_training").toPandas()
validation_df = spark.table("workspace.f1_racing_laptime_pred.raw_validating").toPandas()

print(f"✓ Loaded {len(training_df)} training laps")
print(f"✓ Loaded {len(validation_df)} validation laps")

# Merge function
def merge_circuit_geom_features(df, geom_lookup):
    """
    Merge circuit geometric features into the dataset based on circuit location.
    """
    # Features to add (exclude Circuit_Name since it's the key)
    feature_cols = ['length_official', 'length_computed', 'altitude', 'num_corners',
                    'mean_corner_radius', 'min_corner_radius', 'std_corner_radius',
                    'straight_ratio', 'longest_straight', 'total_turning', 
                    'left_right_ratio', 'bbox_elongation', 'compactness']
    
    # Initialize columns with NaN
    for col in feature_cols:
        df[f'Geom_{col}'] = np.nan
    
    # Map features based on circuit location
    matched = 0
    unmatched_locations = set()
    
    for idx, row in df.iterrows():
        location = row.get('Circuit_Name', None)  # 'Circuit_Name' contains circuit location
        if location and location in geom_lookup:
            geom_data = geom_lookup[location]
            for col in feature_cols:
                if col in geom_data:
                    df.at[idx, f'Geom_{col}'] = geom_data[col]
            matched += 1
        else:
            unmatched_locations.add(location)
    
    print(f"  Matched {matched}/{len(df)} laps with circuit geometric features")
    if unmatched_locations:
        print(f"  Warning: Unmatched locations: {unmatched_locations}")
    
    return df

# Merge geometric features
print("\nMerging circuit geometric features...")
print("Training dataset:")
training_enriched = merge_circuit_geom_features(training_df.copy(), circuit_geom_lookup)

print("\nValidation dataset:")
validation_enriched = merge_circuit_geom_features(validation_df.copy(), circuit_geom_lookup)

# Check for missing values
geom_cols = [col for col in training_enriched.columns if col.startswith('Geom_')]
print(f"\nAdded {len(geom_cols)} geometric feature columns:")
print(geom_cols)

missing_train = training_enriched[geom_cols].isnull().sum().sum()
missing_val = validation_enriched[geom_cols].isnull().sum().sum()
print(f"\nMissing values: Training={missing_train}, Validation={missing_val}")

# Write enriched datasets back to Unity Catalog
print("\n" + "="*60)
print("Writing enriched datasets to Unity Catalog...")
print("="*60)

# ============================================================================
# NORMALIZE CIRCUIT GEOMETRIC FEATURES
# ============================================================================
print("\n" + "="*80)
print("NORMALIZING CIRCUIT GEOMETRIC FEATURES")
print("="*80)

# Based on distribution analysis from cell 6:
# 1. StandardScaler (Gaussian-like, |skewness| < 1.0)
gaussian_geom_cols = ['Geom_length_official', 'Geom_length_computed', 'Geom_num_corners',
                      'Geom_mean_corner_radius', 'Geom_std_corner_radius', 
                      'Geom_longest_straight', 'Geom_total_turning']

# 2. Log1p + StandardScaler (Skewed, |skewness| > 1.0)
skewed_geom_cols = ['Geom_altitude', 'Geom_min_corner_radius', 'Geom_left_right_ratio',
                    'Geom_bbox_elongation', 'Geom_compactness']

# 3. Keep as-is (Ratios/Proportions [0,1])
ratio_geom_cols = ['Geom_straight_ratio']

# Verify all columns exist
gaussian_geom_existing = [c for c in gaussian_geom_cols if c in training_enriched.columns]
skewed_geom_existing = [c for c in skewed_geom_cols if c in training_enriched.columns]
ratio_geom_existing = [c for c in ratio_geom_cols if c in training_enriched.columns]

print(f"\nFound:")
print(f"  - {len(gaussian_geom_existing)} Gaussian geometric features")
print(f"  - {len(skewed_geom_existing)} Skewed geometric features")
print(f"  - {len(ratio_geom_existing)} Ratio geometric features")

# Initialize scalers dictionary for geometric features
geom_scalers_train = {}
geom_scalers_val = {}

# 1. Normalize Gaussian geometric features (StandardScaler)
if gaussian_geom_existing:
    print(f"\nNormalizing {len(gaussian_geom_existing)} Gaussian geometric features...")
    
    # Training scaler
    scaler_gauss_train = StandardScaler()
    training_enriched[gaussian_geom_existing] = scaler_gauss_train.fit_transform(
        training_enriched[gaussian_geom_existing]
    )
    geom_scalers_train['gaussian'] = scaler_gauss_train
    
    # Validation scaler (fit on validation data)
    scaler_gauss_val = StandardScaler()
    validation_enriched[gaussian_geom_existing] = scaler_gauss_val.fit_transform(
        validation_enriched[gaussian_geom_existing]
    )
    geom_scalers_val['gaussian'] = scaler_gauss_val
    
    print(f"  ✓ Applied StandardScaler to {len(gaussian_geom_existing)} features")

# 2. Normalize Skewed geometric features (Log1p + StandardScaler)
if skewed_geom_existing:
    print(f"\nNormalizing {len(skewed_geom_existing)} skewed geometric features...")
    geom_scalers_train['skewed'] = {}
    geom_scalers_val['skewed'] = {}
    
    for col in skewed_geom_existing:
        # Training: log1p transform then StandardScaler
        training_enriched[col] = np.log1p(training_enriched[col])
        scaler_train = StandardScaler()
        training_enriched[[col]] = scaler_train.fit_transform(training_enriched[[col]])
        geom_scalers_train['skewed'][col] = scaler_train
        
        # Validation: log1p transform then StandardScaler
        validation_enriched[col] = np.log1p(validation_enriched[col])
        scaler_val = StandardScaler()
        validation_enriched[[col]] = scaler_val.fit_transform(validation_enriched[[col]])
        geom_scalers_val['skewed'][col] = scaler_val
    
    print(f"  ✓ Applied log1p + StandardScaler to {len(skewed_geom_existing)} features")

# 3. Ratio features - keep as-is (already in [0,1])
if ratio_geom_existing:
    print(f"\nKeeping {len(ratio_geom_existing)} ratio features as-is (already [0,1])")

# Save geometric feature scalers
print("\n" + "="*80)
print("SAVING GEOMETRIC FEATURE SCALERS")
print("="*80)

joblib.dump(geom_scalers_train, 'circuit_geom_scalers_train.joblib')
print("✓ Saved training geometric scalers to circuit_geom_scalers_train.joblib")

joblib.dump(geom_scalers_val, 'circuit_geom_scalers_val.joblib')
print("✓ Saved validation geometric scalers to circuit_geom_scalers_val.joblib")

print("\nScaler contents:")
print(f"  Training scalers: {list(geom_scalers_train.keys())}")
print(f"  Validation scalers: {list(geom_scalers_val.keys())}")

# Write enriched and normalized datasets back to Unity Catalog
print("\n" + "="*80)
print("WRITING ENRICHED DATASETS TO UNITY CATALOG")
print("="*80)

spark_train_enriched = spark.createDataFrame(training_enriched)
spark_train_enriched = spark_train_enriched.drop("length_official", "length_computed", "altitude", "num_corners", "AverageAngleAbs", "AverageAngle")
spark_train_enriched.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("workspace.f1_racing_laptime_pred.raw_training")
print(f"✓ Saved {len(training_enriched)} training laps with normalized geometric features")

spark_val_enriched = spark.createDataFrame(validation_enriched)
spark_val_enriched = spark_val_enriched.drop("length_official", "length_computed", "altitude", "num_corners", "AverageAngleAbs", "AverageAngle")
spark_val_enriched.write.mode("overwrite").option("mergeSchema", "true")Compound.saveAsTable("workspace.f1_racing_laptime_pred.raw_validating")
print(f"✓ Saved {len(validation_enriched)} validation laps with normalized geometric features")

print("\n" + "="*60)
print("COMPLETE! Circuit geometric features merged successfully")
print("="*60)

In [0]:

'''


def circuit_info(race):
        avg=[]
        std=[]
        circuit = race.get_circuit_info()
        total_angle = 0
        count = 0
        for angle in circuit.corners['Angle']:
            total_angle = total_angle + angle
            count = count + 1
        
        average = total_angle/count
        avg.append(average)

        std_sum=0
        for angle in circuit.corners['Angle']:
            std_sum = std_sum + (angle - average)*(angle - average)
        
        std_deviation = math.sqrt(std_sum/(count-1))
        std.append(std_deviation)

- embedded encoding driver and team
- one hot encoding for tyre compound
'''